# Notebook 02: The Bayesian Update Loop

## Why This Matters

Most introductions to Bayesian inference treat Bayes' theorem as a formula you apply once. But the real power is in repetition: every posterior becomes the prior for the next observation. This sequential update loop is what makes Bayesian inference fundamentally different from frequentist batch analysis.

It's also what makes Bayesian methods natural for streaming data, online learning, and decision-making under uncertainty. A frequentist approach requires collecting all the data before running a test. A Bayesian can update their beliefs with each new observation, at any point decide if they have enough information, and stop when the decision is clear — without inflating the false positive rate through optional stopping. (This is the foundation of notebook 14's sequential testing.)

The key machinery that makes sequential updating tractable is **conjugacy**: special pairs of prior distributions and likelihood functions where the posterior has the same family as the prior. With conjugate models, updating is just adding numbers — no MCMC required. Understanding conjugate models builds the intuition you need to read what PyMC is doing under the hood.

## Background

### Conjugate Families — Narrative

A prior distribution is **conjugate** to a likelihood if the posterior is in the same distributional family as the prior. This isn't magic — it falls out of algebra. The three conjugate pairs you'll use constantly:

**Beta-Binomial**: The workhorse for A/B testing and conversion rates.
- Prior: `Beta(α, β)` — encodes your prior beliefs about a probability p
- Likelihood: `Binomial(n, p)` — observe k successes in n trials
- Posterior: `Beta(α + k, β + n - k)` — just add the counts!
- The hypers α and β are *pseudo-counts*: prior successes and prior failures

**Normal-Normal**: The workhorse for estimating means.
- Prior: `Normal(μ₀, σ₀)` on the unknown mean
- Likelihood: `Normal(μ, σ_known)` — observe data with known noise
- Posterior: precision-weighted combination of prior and data means
- The posterior mean is a *shrinkage estimator*: it pulls the data mean toward the prior

**Poisson-Gamma**: The workhorse for count data and rates.
- Prior: `Gamma(α, β)` on the unknown rate λ
- Likelihood: `Poisson(λ)` — observe n_events in total_exposure time
- Posterior: `Gamma(α + n_events, β + total_exposure)`

### Why Conjugacy Matters in Practice

Conjugate models are fast (closed-form), interpretable (pseudo-counts), and exact. MCMC approximates the posterior through sampling — for simple models, why use a hammer when a screwdriver works? The practical rule: use conjugate models when your likelihood and prior match; fall back to PyMC/MCMC when they don't (i.e., most hierarchical and non-linear models).

### Gibbs Sampling — Why We Don't Implement It for Production

Gibbs sampling is a classic MCMC algorithm: cycle through parameters one at a time, drawing each from its full conditional given all others. It works beautifully for conjugate models (each full conditional is in a known family), which is why it dominated Bayesian computation from the 1990s through the mid-2000s.

But Gibbs has two problems that made it obsolete for most modern use cases:
1. **Slow mixing in high dimensions**: Parameters are updated one at a time, so correlated parameters explore the space very slowly
2. **Requires deriving full conditionals**: For non-conjugate models, this is analytically intractable

Hamiltonian Monte Carlo (HMC), covered in notebook 03, uses gradient information to take large, geometry-aware steps in the joint parameter space. NUTS (No-U-Turn Sampler), the adaptive variant implemented in PyMC and Stan, made HMC practical without tuning. For the same computational budget, HMC mixes 10-1000x faster than Gibbs on typical Bayesian models.

**The bottom line**: understand Gibbs conceptually (it's in every Bayesian textbook), but use HMC/NUTS in practice.

## Structure of This Notebook

1. Beta-Binomial: coin flip and conversion rate estimation
2. Normal-Normal: mean estimation with known variance
3. Poisson-Gamma: rate estimation for count data
4. The update loop in action: streaming data simulation
5. Comparing conjugate analytical posteriors to Monte Carlo estimates

## What You'll Learn

- Implementing the three main conjugate update rules
- Why the posterior is a precision-weighted average in the Normal-Normal case
- How the posterior collapses around the true value as n grows
- That sampling from the posterior approximates what conjugate updates give exactly

In [ ]:
%pip install numpy scipy matplotlib --quiet

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from dataclasses import dataclass

rng = np.random.default_rng(42)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
print('Ready.')

## 1. Beta-Binomial — Conversion Rate Estimation

In [ ]:
@dataclass
class BetaBinomialPosterior:
    alpha: float
    beta: float

    @property
    def dist(self):
        return stats.beta(self.alpha, self.beta)

    @property
    def mean(self):
        return self.alpha / (self.alpha + self.beta)

    @property
    def mode(self):
        if self.alpha > 1 and self.beta > 1:
            return (self.alpha - 1) / (self.alpha + self.beta - 2)
        return None

    def credible_interval(self, level=0.95):
        q = (1 - level) / 2
        return self.dist.ppf(q), self.dist.ppf(1 - q)

    def update(self, n_successes: int, n_trials: int) -> 'BetaBinomialPosterior':
        """Conjugate update: returns new posterior after observing data."""
        return BetaBinomialPosterior(
            alpha=self.alpha + n_successes,
            beta=self.beta + (n_trials - n_successes)
        )

    def prob_greater_than(self, threshold: float) -> float:
        """P(θ > threshold) — useful for decision making."""
        return 1 - self.dist.cdf(threshold)

    def prob_a_better_than_b(self, other: 'BetaBinomialPosterior', n_samples=50_000) -> float:
        """P(θ_a > θ_b) via Monte Carlo from two independent posteriors."""
        samples_a = self.dist.rvs(n_samples)
        samples_b = other.dist.rvs(n_samples)
        return (samples_a > samples_b).mean()


# Scenario: estimating e-commerce conversion rate
# Industry average is ~3%, we encode this weakly
prior = BetaBinomialPosterior(alpha=2, beta=65)  # prior mean ≈ 3%

# Observe data in batches (visitors arriving over time)
batches = [(200, 5), (300, 11), (500, 18), (400, 8), (600, 22)]

theta_grid = np.linspace(0.001, 0.12, 500)
fig, axes = plt.subplots(2, 3, figsize=(13, 7))

# Prior
ax = axes[0, 0]
ax.plot(theta_grid, prior.dist.pdf(theta_grid), 'steelblue', lw=2)
ax.fill_between(theta_grid, prior.dist.pdf(theta_grid), alpha=0.2)
ci = prior.credible_interval()
ax.set_title(f'Prior\nBeta({prior.alpha:.0f},{prior.beta:.0f})\nmean={prior.mean:.3f}, '
             f'CI=[{ci[0]:.3f},{ci[1]:.3f}]', fontsize=8)
ax.set_xlabel('Conversion rate θ')
ax.grid(True, alpha=0.3)

posterior = prior
total_visitors, total_conversions = 0, 0

for i, (visitors, conversions) in enumerate(batches):
    posterior = posterior.update(conversions, visitors)
    total_visitors += visitors
    total_conversions += conversions
    ax = axes[(i + 1) // 3, (i + 1) % 3]
    ax.plot(theta_grid, posterior.dist.pdf(theta_grid), 'steelblue', lw=2)
    ax.fill_between(theta_grid, posterior.dist.pdf(theta_grid), alpha=0.2)
    ci = posterior.credible_interval()
    ax.axvline(posterior.mean, color='red', linestyle='--', lw=1.5)
    ax.set_title(
        f'+{visitors} visitors (+{conversions} conv.)\n'
        f'Total: {total_visitors} vis, {total_conversions} conv\n'
        f'mean={posterior.mean:.4f}, CI=[{ci[0]:.3f},{ci[1]:.3f}]',
        fontsize=8
    )
    ax.set_xlabel('Conversion rate θ')
    ax.grid(True, alpha=0.3)

plt.suptitle('Beta-Binomial: Conversion Rate Posterior Updates', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Final posterior: Beta({posterior.alpha:.0f}, {posterior.beta:.0f})')
print(f'Posterior mean: {posterior.mean:.4f} ({posterior.mean*100:.2f}%)')
print(f'95% credible interval: {[f"{x:.4f}" for x in posterior.credible_interval()]}')
print(f'P(θ > 3%): {posterior.prob_greater_than(0.03):.4f}')
print(f'P(θ > 4%): {posterior.prob_greater_than(0.04):.4f}')

## 2. Normal-Normal — Mean Estimation with Known Variance

In [ ]:
@dataclass
class NormalNormalPosterior:
    """
    Normal-Normal conjugate model.
    Estimates unknown mean μ given known observation noise σ_obs.

    Posterior update (precision-weighted):
      precision = 1/variance
      posterior_precision = prior_precision + n/σ_obs²
      posterior_mean = (prior_precision * μ₀ + (n/σ_obs²) * x̄) / posterior_precision
    """
    mu: float        # prior/posterior mean
    sigma: float     # prior/posterior std
    sigma_obs: float # observation noise (assumed known)

    @property
    def precision(self):
        return 1 / self.sigma**2

    @property
    def dist(self):
        return stats.norm(self.mu, self.sigma)

    def credible_interval(self, level=0.95):
        q = (1 - level) / 2
        return self.dist.ppf(q), self.dist.ppf(1 - q)

    def update(self, observations: np.ndarray) -> 'NormalNormalPosterior':
        """Conjugate update given new observations (assumed i.i.d. Normal)."""
        n = len(observations)
        x_bar = observations.mean()

        data_precision = n / self.sigma_obs**2
        posterior_precision = self.precision + data_precision
        posterior_sigma = 1 / np.sqrt(posterior_precision)
        posterior_mu = (
            self.precision * self.mu + data_precision * x_bar
        ) / posterior_precision

        return NormalNormalPosterior(posterior_mu, posterior_sigma, self.sigma_obs)


# Scenario: estimating mean checkout time for an e-commerce site
# We know observation noise is ~30 seconds (from historical data)
true_mu = 180  # true mean: 180 seconds
sigma_obs = 30  # observation noise: 30 seconds

prior = NormalNormalPosterior(mu=200, sigma=50, sigma_obs=sigma_obs)

# Simulate batches of observations
n_batches = 5
batch_size = 10
observations = rng.normal(true_mu, sigma_obs, n_batches * batch_size)

x_range = np.linspace(50, 350, 500)
fig, axes = plt.subplots(2, 3, figsize=(13, 7))

ax = axes[0, 0]
ax.plot(x_range, prior.dist.pdf(x_range), 'steelblue', lw=2)
ax.fill_between(x_range, prior.dist.pdf(x_range), alpha=0.2)
ax.axvline(true_mu, color='red', linestyle='--', lw=1.5, label=f'True μ={true_mu}')
ci = prior.credible_interval()
ax.set_title(f'Prior N({prior.mu},{prior.sigma})\nCI=[{ci[0]:.0f},{ci[1]:.0f}]', fontsize=9)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

posterior = prior
total_obs = 0

for i in range(n_batches):
    batch = observations[i*batch_size:(i+1)*batch_size]
    posterior = posterior.update(batch)
    total_obs += batch_size
    ax = axes[(i+1)//3, (i+1)%3]
    ax.plot(x_range, posterior.dist.pdf(x_range), 'steelblue', lw=2)
    ax.fill_between(x_range, posterior.dist.pdf(x_range), alpha=0.2)
    ax.axvline(true_mu, color='red', linestyle='--', lw=1.5, label=f'True μ={true_mu}')
    ax.axvline(posterior.mu, color='green', linestyle=':', lw=1.5,
               label=f'Post mean={posterior.mu:.1f}')
    ci = posterior.credible_interval()
    ax.set_title(
        f'+{batch_size} obs (n={total_obs} total)\n'
        f'mean={posterior.mu:.1f}s, σ={posterior.sigma:.1f}s\n'
        f'CI=[{ci[0]:.0f},{ci[1]:.0f}]',
        fontsize=8
    )
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle('Normal-Normal: Mean Estimation via Precision-Weighted Updates', fontsize=12)
plt.tight_layout()
plt.show()

print('Precision-weighted update mechanics:')
print(f'  Prior precision: {prior.precision:.5f}')
print(f'  Data precision per obs: {1/sigma_obs**2:.5f}')
print(f'  After {total_obs} obs: data contributes {total_obs/sigma_obs**2:.4f} total precision')
print(f'  → posterior dominated by data when n/σ_obs² >> prior precision')
print(f'  → posterior dominated by prior when n is small')

## 3. Poisson-Gamma — Rate Estimation for Count Data

In [ ]:
@dataclass
class PoissonGammaPosterior:
    """
    Poisson-Gamma conjugate model for estimating event rates.

    Prior: λ ~ Gamma(α, β)  (β = rate parameter, mean = α/β)
    Likelihood: counts ~ Poisson(λ * exposure)
    Posterior: λ ~ Gamma(α + total_events, β + total_exposure)

    Use cases:
    - Website errors per hour
    - Customer support tickets per day
    - Defects per unit manufactured
    """
    alpha: float  # shape
    beta: float   # rate (not scale!)

    @property
    def dist(self):
        return stats.gamma(self.alpha, scale=1/self.beta)

    @property
    def mean(self):
        return self.alpha / self.beta

    def credible_interval(self, level=0.95):
        q = (1 - level) / 2
        return self.dist.ppf(q), self.dist.ppf(1 - q)

    def update(self, total_events: int, total_exposure: float) -> 'PoissonGammaPosterior':
        """Conjugate update: add events to alpha, add exposure to beta."""
        return PoissonGammaPosterior(
            alpha=self.alpha + total_events,
            beta=self.beta + total_exposure
        )

    def predictive_mean(self, future_exposure: float) -> float:
        """Expected events in next `future_exposure` units."""
        return self.mean * future_exposure

    def predictive_interval(self, future_exposure: float, level=0.95):
        """Posterior predictive interval for future counts.
        Uses NegBin(alpha, beta/(beta+exposure)) — marginalizes out λ."""
        p = self.beta / (self.beta + future_exposure)
        nb = stats.nbinom(self.alpha, p)
        q = (1 - level) / 2
        return nb.ppf(q), nb.ppf(1 - q)


# Scenario: estimating API error rate (errors per 1000 requests)
true_rate = 2.3  # true errors per 1000 requests

# Prior: expect ~2 errors per 1000 requests based on historical data, moderate confidence
prior = PoissonGammaPosterior(alpha=4, beta=2)  # mean=2, some uncertainty

# Simulate daily observations over 2 weeks
# Each day: ~5000 requests, observe counts
daily_requests = 5  # in units of 1000
n_days = 14
daily_counts = rng.poisson(true_rate * daily_requests, n_days)

print(f'True error rate: {true_rate} per 1000 requests')
print(f'Daily observations (counts per {daily_requests}k requests):')
print(f'  {daily_counts}')
print(f'  Mean: {daily_counts.mean():.2f}, Expected: {true_rate * daily_requests:.2f}')

lambda_range = np.linspace(0.01, 6, 500)
fig, ax = plt.subplots(figsize=(10, 5))

posterior = prior
colors = plt.cm.Blues(np.linspace(0.3, 1.0, n_days + 1))

ax.plot(lambda_range, prior.dist.pdf(lambda_range), color=colors[0], lw=2,
        label=f'Prior (mean={prior.mean:.1f})')

for day, count in enumerate(daily_counts):
    posterior = posterior.update(count, daily_requests)
    if (day + 1) in [1, 3, 7, 14]:
        ax.plot(lambda_range, posterior.dist.pdf(lambda_range),
                color=colors[day + 1], lw=2,
                label=f'Day {day+1} (n={day+1}, mean={posterior.mean:.2f})')

ax.axvline(true_rate, color='red', linestyle='--', lw=2, label=f'True rate={true_rate}')
ax.set_xlabel('Error rate λ (per 1000 requests)')
ax.set_ylabel('Posterior density')
ax.set_title('Poisson-Gamma: API Error Rate — Sequential Updates')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

ci = posterior.credible_interval()
pred_ci = posterior.predictive_interval(future_exposure=5, level=0.95)
print(f'\nFinal posterior: Gamma({posterior.alpha:.0f}, {posterior.beta:.0f})')
print(f'Posterior mean: {posterior.mean:.3f} errors/1000 requests')
print(f'95% credible interval: [{ci[0]:.3f}, {ci[1]:.3f}]')
print(f'Predictive 95% CI for next day (5k requests): {pred_ci}')

## 4. The Update Loop — Conjugate vs. Monte Carlo

In [ ]:
# Show that Monte Carlo sampling from the posterior converges to the
# analytical conjugate result. This builds the bridge to MCMC in notebook 03.

def monte_carlo_posterior(n_samples_prior, prior_alpha, prior_beta, n_heads, n_trials,
                          n_mc_samples=100_000):
    """
    Approximate the Beta-Binomial posterior via Monte Carlo:
    1. Sample θ from the prior
    2. Compute P(data | θ) for each sample
    3. Resample weighted by likelihood (importance sampling)
    """
    # Prior samples
    theta_samples = stats.beta(prior_alpha, prior_beta).rvs(n_mc_samples)

    # Log likelihood under Binomial
    log_likelihood = (
        n_heads * np.log(theta_samples) +
        (n_trials - n_heads) * np.log(1 - theta_samples)
    )
    log_w = log_likelihood - log_likelihood.max()
    w = np.exp(log_w)
    w /= w.sum()

    # Resample from prior weighted by likelihood
    posterior_samples = rng.choice(theta_samples, size=n_mc_samples, p=w)
    return posterior_samples


# True posterior parameters
prior_a, prior_b = 3, 10
n_heads, n_trials = 25, 60
post_a = prior_a + n_heads
post_b = prior_b + (n_trials - n_heads)

analytical_dist = stats.beta(post_a, post_b)
mc_samples = monte_carlo_posterior(
    n_mc_samples := 100_000, prior_a, prior_b, n_heads, n_trials
)

theta_grid = np.linspace(0.01, 0.99, 400)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(theta_grid, analytical_dist.pdf(theta_grid), 'b-', lw=2.5,
             label=f'Analytical: Beta({post_a},{post_b})')
axes[0].hist(mc_samples, bins=80, density=True, alpha=0.5, color='orange',
             label=f'Monte Carlo ({n_mc_samples:,} samples)')
axes[0].set_xlabel('θ')
axes[0].set_ylabel('Density')
axes[0].set_title('Analytical vs. Monte Carlo Posterior')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Credible intervals comparison
analytical_ci = analytical_dist.ppf([0.025, 0.975])
mc_ci = np.percentile(mc_samples, [2.5, 97.5])

axes[1].barh(['MC\n(importance sampling)', 'Analytical\n(conjugate)'],
             [mc_ci[1] - mc_ci[0], analytical_ci[1] - analytical_ci[0]],
             left=[mc_ci[0], analytical_ci[0]],
             height=0.4, color=['orange', 'steelblue'], alpha=0.8)
axes[1].set_xlabel('θ (probability of heads)')
axes[1].set_title('95% Credible Interval Comparison')
axes[1].grid(True, alpha=0.3, axis='x')

print(f'Analytical 95% CI: [{analytical_ci[0]:.4f}, {analytical_ci[1]:.4f}]')
print(f'MC 95% CI:         [{mc_ci[0]:.4f}, {mc_ci[1]:.4f}]')
print(f'Analytical mean: {analytical_dist.mean():.4f}')
print(f'MC mean:         {mc_samples.mean():.4f}')

plt.tight_layout()
plt.show()

print('\nKey insight: importance sampling approximates conjugate results.')
print('But it breaks down in high dimensions — that is exactly why we need MCMC (notebook 03).')

## 5. Bayesian Shrinkage — The Regularization Connection

In [ ]:
# Bayesian shrinkage: the posterior mean is a weighted average of prior and data
# This is mathematically equivalent to regularization in the frequentist world.
# Normal-Normal with N(0,τ²) prior on μ gives:
# posterior mean = x_bar * (n/σ²) / (n/σ² + 1/τ²)
# This is the Ridge regression estimator when applied to regression coefficients!

true_theta = 0.35
sigma_obs = 0.3  # observation noise
prior_mean = 0.5  # prior mean
prior_sigma = 0.2  # prior uncertainty

sample_sizes = np.arange(1, 201)
posterior_means = []
mle_means = []

for n in sample_sizes:
    obs = rng.normal(true_theta, sigma_obs, n)
    prior = NormalNormalPosterior(prior_mean, prior_sigma, sigma_obs)
    posterior = prior.update(obs)
    posterior_means.append(posterior.mu)
    mle_means.append(obs.mean())

# Smooth for display
from scipy.ndimage import uniform_filter1d
smooth_mle = uniform_filter1d(mle_means, size=10)

fig, ax = plt.subplots(figsize=(10, 5))
ax.axhline(true_theta, color='black', lw=2, linestyle='--', label=f'True θ={true_theta}')
ax.axhline(prior_mean, color='gray', lw=1.5, linestyle=':', label=f'Prior mean={prior_mean}')
ax.plot(sample_sizes, smooth_mle, 'orange', lw=1.5, alpha=0.8, label='MLE (data mean, smoothed)')
ax.plot(sample_sizes, posterior_means, 'steelblue', lw=2, label='Posterior mean (Bayesian)')
ax.set_xlabel('Number of Observations')
ax.set_ylabel('Estimate of θ')
ax.set_title('Bayesian Shrinkage: Prior Dominates Early, Data Dominates Late\n'
             'Posterior mean = precision-weighted average of prior and MLE')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Analytical shrinkage formula
print('Shrinkage formula (Normal-Normal):')
print('  posterior_mean = λ * x̄ + (1-λ) * μ₀')
print('  where λ = (n/σ_obs²) / (n/σ_obs² + 1/σ₀²)')
for n in [1, 5, 20, 100]:
    data_prec = n / sigma_obs**2
    prior_prec = 1 / prior_sigma**2
    lam = data_prec / (data_prec + prior_prec)
    print(f'  n={n:3d}: λ={lam:.3f} → {lam*100:.1f}% data, {(1-lam)*100:.1f}% prior')

## Summary

### The Three Conjugate Families

| Model | Prior | Likelihood | Posterior | Update Rule |
|-------|-------|-----------|-----------|-------------|
| Beta-Binomial | Beta(α,β) | Binomial(n,p) | Beta(α+k, β+n-k) | Add counts |
| Normal-Normal | Normal(μ₀,σ₀) | Normal(μ,σ_obs) | Precision-weighted avg | Precision-weighted mean |
| Poisson-Gamma | Gamma(α,β) | Poisson(λt) | Gamma(α+events, β+exposure) | Add events & exposure |

### Key Takeaways

- **Conjugacy makes updating trivial**: just add pseudo-counts or precision. No sampling required.
- **Bayesian shrinkage = regularization**: the posterior mean is always pulled toward the prior, more strongly with small data, less with large data.
- **Monte Carlo approximates conjugate results** — this is the conceptual bridge to MCMC. When the posterior isn't analytically tractable, we sample instead.
- **Gibbs sampling** (sample each parameter from its full conditional) works elegantly for conjugate models but is slow for correlated high-dimensional posteriors. HMC is the modern answer.
- **The posterior predictive** integrates out parameter uncertainty: instead of predicting with a single θ, you average predictions across the posterior distribution of θ.

### The Limits of Conjugacy

Conjugate priors only exist for exponential family likelihoods. Real models — logistic regression, multilevel models, GPs, BNNs — don't have conjugate priors. For everything beyond the three models above, we need numerical methods. That's notebook 03 (MCMC) and 04 (PyMC).

Next: Notebook 03 covers MCMC — the Metropolis-Hastings algorithm (for intuition), and HMC/NUTS via PyMC (for everything practical), with diagnostics for chain convergence.